In [1]:
%cd ../../

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
import time
import random
import json
from itertools import combinations, product
import uuid

import pandas as pd
import polars as pl

from src.forecaster import ModelService

In [3]:
seed = time.time()
random.seed(seed)

In [4]:
restaurant = 1
schoolyear = "24-25"
date_start = "2025-06-02"

# Load tables

In [5]:
path = "data/processed/dim_meal_types.xlsx"

dim_meal_types = pl.read_excel(path)
dim_meal_types

meal_type_id,meal_type,meal_type_en
i64,str,str
1,"""Kala""","""fish"""
2,"""Liha""","""meat"""
3,"""Kana""","""chicken"""
4,"""Vegaani""","""vegan"""
5,"""Kasvis""","""vegetarian"""
6,"""Buffet""","""buffet"""
7,"""Not Mapped""","""not_mapped"""


In [6]:
path = "data/processed/dim_meals.parquet"
dim_meals = (
    pl.read_parquet(path)
    # .drop('meal_codes', 'names', 'src')

    # .join(
    #     dim_meal_types.select('meal_type_id', 'meal_type_en'),
    #     left_on='meal_type', right_on='meal_type_id',
    #     how='left'
    # )
)
dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


### Load list of selected meals specific for each restaurant and weekday

File `recommended_meal_list.json` has following structure

```json
{
    "<restaurant_id>": {
        "<weekday_id>": {
            "non-vegan": [<list of meal id>],
            "vegan": [<list of meal id>],
        }
    }
}
```

In [7]:
path = "data/processed/recommended_meal_list.json"
with open(path) as file:
    meals_by_day_raw = json.load(file)

meals_by_day = {}
for restau, weekdays in meals_by_day_raw.items():
    meals_by_day[int(restau)] = {}
    for weekday, meals in weekdays.items():
        meals_by_day[int(restau)][int(weekday)] = meals

# Craft menus

In [8]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = 3
MAX_MEAL_OCCURENCES = 2
NUM_FISH_PER_WEEK = 2

NUM_DAY_LEVEL_MENUS = 1_000_000
NUM_WEEK_LEVEL_MENUS = 1000
NUM_MENUS_FINAL = 20

THETA_CO2 = 0.5
THETA_WASTE = 0.04
THETA_KELA = 2
THETA_GLUTEN = 1

ALPHA_POS = 2
ALPHA_CO2 = 1
ALPHA_WASTE = 1
ALPHA_KELA = 2
ALPHA_GLUTEN = 2

## Craft the day-level menus

Day-level menus must satisfy:
- condition (2), (6) and (9)
- containing a variety of meals

***[June 3, 2025]***
- Condition (1.2) and (8) changed from strict to loose

In [9]:
def craft_day_level_menu(meals_by_day: dict, restaurant: int, weekday: int, n_max: int = 10_000_000) -> list:
    assert restaurant in meals_by_day and weekday in meals_by_day[restaurant]
    non_vegan = meals_by_day[restaurant][weekday]['non-vegan']
    vegan = meals_by_day[restaurant][weekday]['vegan']

    combos = [[x[0], *x[1]] for x in product(non_vegan, combinations(vegan, NUM_VEGAN_PER_DAY))]
    
    return combos[:n_max]

In [10]:
combos_mon = craft_day_level_menu(meals_by_day, restaurant, 1)
combos_tue = craft_day_level_menu(meals_by_day, restaurant, 2)
combos_wed = craft_day_level_menu(meals_by_day, restaurant, 3)
combos_thu = craft_day_level_menu(meals_by_day, restaurant, 4)
combos_fri = craft_day_level_menu(meals_by_day, restaurant, 5)

# Craft week-level menu

Week-level menus must satisfy:
- condition (1.1), (3)

In [11]:
meal_type_fish = dim_meal_types.filter(pl.col('meal_type_en') == pl.lit('fish'))['meal_type_id'].head().item()

# Create table containing week menu candidates
menus_week = pl.DataFrame({
    '1': random.choices(combos_mon, k=NUM_DAY_LEVEL_MENUS),
    '2': random.choices(combos_tue, k=NUM_DAY_LEVEL_MENUS),
    '3': random.choices(combos_wed, k=NUM_DAY_LEVEL_MENUS),
    '4': random.choices(combos_tue, k=NUM_DAY_LEVEL_MENUS),
    '5': random.choices(combos_fri, k=NUM_DAY_LEVEL_MENUS),
    'weeklevel_idx': pl.Series([str(uuid.uuid4()) for _ in range(NUM_DAY_LEVEL_MENUS)])
})

menus_week.head()

1,2,3,4,5,weeklevel_idx
list[i64],list[i64],list[i64],list[i64],list[i64],str
"[64, 39, 484]","[152, 365, 545]","[528, 284, 483]","[73, 13, 176]","[100, 249, 284]","""78fece6b-5d01-4cb0-a496-993517…"
"[154, 34, 37]","[62, 6, 365]","[27, 12, 117]","[132, 114, 197]","[537, 112, 295]","""84bf34da-b1f6-4dde-a05c-09801e…"
"[55, 229, 289]","[118, 50, 382]","[191, 69, 79]","[149, 101, 135]","[100, 39, 393]","""3998c9fc-fd6d-4ccd-876f-4835cb…"
"[154, 168, 494]","[312, 68, 535]","[132, 112, 483]","[41, 249, 410]","[136, 201, 487]","""0948d700-8756-4be5-b0c9-27757e…"
"[78, 58, 138]","[221, 317, 473]","[147, 105, 229]","[116, 69, 176]","[100, 26, 196]","""9dd92e34-4a46-469f-9a19-807355…"


In [13]:
ids_valid_week_menu = (
    menus_week
    .select(
        'weeklevel_idx',
        pl.concat_list(["1", "2", "3", "4", "5"]).alias('meal')
    )
    .explode('meal')


    # For each week menu, find the max occurence of meals in the week menu
    .with_columns(
        pl.len().over('weeklevel_idx', 'meal').alias('count_occurence'),
    )
    .with_columns(
        pl.col('count_occurence').max().over('weeklevel_idx').alias('count_max_occurence')
    )

    # Remove week menu candidates not satisfying (3)
    .filter(pl.col('count_max_occurence') <= MAX_MEAL_OCCURENCES)
    


    # Add meal type info and count no. fish meals of each week menu candidate
    .join(dim_meals.select('id', 'meal_type'), left_on='meal', right_on='id', how='left')
    .with_columns(
        (pl.col('meal_type') == meal_type_fish).cast(pl.Int32).alias('is_fish')
    )
    .with_columns(
        pl.col('is_fish').sum().over('weeklevel_idx').alias('count_fish')
    )

    # Remove week menu candidates not satisfying (1.1)
    .filter(pl.col('count_fish') >= NUM_FISH_PER_WEEK)

    .select('weeklevel_idx')
    .unique()
)

menus_week = (
    menus_week
    .join(ids_valid_week_menu, on='weeklevel_idx', how='inner')
    .sample(NUM_WEEK_LEVEL_MENUS)

    .melt(
        'weeklevel_idx',
        value_vars=["1", "2", "3", "4", "5"],
        variable_name="weekday",
        value_name="meal"
    )

    .with_columns(pl.col('weekday').cast(pl.Int32))
)
menus_week.head()

weeklevel_idx,weekday,meal
str,i32,list[i64]
"""84b9ba2c-a63d-4477-a6a2-efa4ec…",1,"[55, 106, 186]"
"""48e99a81-bf60-461d-93c7-6ff6ac…",1,"[32, 79, 475]"
"""a4581402-78e0-47fa-b853-d1e520…",1,"[64, 50, 317]"
"""825e8013-3f72-4dc5-ac9f-5a0a8c…",1,"[543, 201, 491]"
"""f696c48b-bea3-4bf9-821c-8a895d…",1,"[64, 39, 400]"


# Add meal-specific info and date-specific needed for calculating score

Following info will be added:
- `whole_pos` (forecasted)
- `whole_waste` (forecasted)
- meal's CO2
- meal's POS (forecasted)
- meal's gluten
- meal's kela

In [14]:
model = ModelService()

In [15]:
weekday2date = pl.DataFrame({
    'weekday': [1, 2, 3, 4, 5],
    'date': pl.Series(pd.date_range(date_start, periods=5)).dt.date()
})
weekday2date.head()

weekday,date
i64,date
1,2025-06-02
2,2025-06-03
3,2025-06-04
4,2025-06-05
5,2025-06-06


In [16]:
last_date = weekday2date['date'].max().strftime(r"%Y-%m-%d")
pos_restaurant = (
    pl.from_dataframe(model.forecast_pos_restaurant(restaurant, last_date))
    .select(
        pl.col('date').dt.date(),
        pl.col('forecasted').alias('whole_pos')
    )
)
waste_restaurant = (
    pl
    .from_dataframe(model.forecast_waste_restaurant(restaurant, last_date))
    .select(
        pl.col('date').dt.date(),
        pl.col('forecasted').alias('whole_waste')
    )
)

In [17]:
s_gluten = "gluten_free"
s_kela = "kela"

menus_week = (
    menus_week
    .explode('meal')

    
    .join(weekday2date, on='weekday')
    .join(dim_meals.select('id', 'meal_type'), left_on='meal', right_on='id', how='left')
    .with_columns(pl.col('date').dt.strftime(r"%Y-%m-%d").alias('date_str'))

    # Forecast POS for each meal
    .with_columns(
        pl.struct('meal', 'date_str', 'meal_type')
        .map_elements(
            lambda r: model.forecast_pos_per_meal(restaurant, r['meal'], r['date_str'], r['meal_type']),
            return_dtype=pl.Float32
        ).alias('pos')
    )
    .drop('date_str')

    # Add forecasted restaurant's waste and pos
    .join(pos_restaurant, on='date', how='left')
    .join(waste_restaurant, on='date', how='left')


    # Add CO2, gluten-free and kela for each meal
    .join(
        dim_meals.select(
            'id', 'co2',
            pl.col('attributes').list.contains(s_gluten).alias('is_gluten').cast(pl.Int32),
            pl.col('attributes').list.contains(s_kela).alias('is_kela').cast(pl.Int32),
        ),
        left_on='meal', right_on='id', how='left'
    )
)


menus_week.head()

weeklevel_idx,weekday,meal,date,meal_type,pos,whole_pos,whole_waste,co2,is_gluten,is_kela
str,i32,i64,date,i64,f32,f64,f64,f32,i32,i32
"""84b9ba2c-a63d-4477-a6a2-efa4ec…",1,55,2025-06-02,2,341.666656,833.382401,39.272509,0.9,1,1
"""84b9ba2c-a63d-4477-a6a2-efa4ec…",1,106,2025-06-02,4,126.0,833.382401,39.272509,0.45,1,1
"""84b9ba2c-a63d-4477-a6a2-efa4ec…",1,186,2025-06-02,5,7.0,833.382401,39.272509,0.0,0,0
"""48e99a81-bf60-461d-93c7-6ff6ac…",1,32,2025-06-02,2,536.666687,833.382401,39.272509,0.63,0,1
"""48e99a81-bf60-461d-93c7-6ff6ac…",1,79,2025-06-02,4,133.0,833.382401,39.272509,0.42,1,1


# Calculate fitness value

In [20]:
menus_week = (
    menus_week

    # Calculate score for each date
    .group_by('weeklevel_idx', 'weekday')
    .agg(
        pl.concat_list(pl.struct('meal', 'pos')).flatten().alias('meals_planned'),

        pl.col('pos').sum().alias('sum_pos'),
        (pl.col('pos') * pl.col('co2')).sum().alias('sum_co2_pos'),
        pl.col('is_gluten').sum().alias('sum_gluten'),
        pl.col('is_kela').sum().alias('sum_kela'),

        pl.col('whole_pos').first(),
        pl.col('whole_waste').first(),
    )

    .with_columns(
        (
            ALPHA_POS * (pl.col('sum_pos') / pl.col('whole_pos') - 1).abs()
            + ALPHA_CO2 * (pl.col('sum_co2_pos') / pl.col('sum_pos') / THETA_CO2)
            + ALPHA_WASTE * (pl.col('whole_waste') / pl.col('sum_pos') / THETA_WASTE)
            + ALPHA_GLUTEN * (1 - pl.col('sum_gluten') / THETA_GLUTEN).clip(0)
            + ALPHA_KELA * (1 - pl.col('sum_kela') / THETA_GLUTEN).clip(0)
        ).alias('score_day')
    )

    # Calculate score for entire week
    .with_columns(
        pl.col('score_day').sum().over('weeklevel_idx').alias('score_week')
    )
    .with_columns(
        pl.col('score_week').rank('dense', descending=False).over(None).alias('rank')
    )
    .filter(pl.col('rank') <= NUM_MENUS_FINAL)
    .sort('rank')


    # Keep columns as data model
    .join(weekday2date, on='weekday', how='left')
    .select(
        pl.concat_str(
            [
                pl.col('weeklevel_idx'),
                pl.col('date').dt.strftime(r"%Y-%m-%d"), 
                pl.lit(restaurant)
            ],
            separator='|'
        ).alias('id'),
        'weeklevel_idx',
        'date',
        'whole_waste',
        'whole_pos',
        'score_week',
        'meals_planned'
    )
)

menus_week.head()

id,weeklevel_idx,date,whole_waste,whole_pos,score_week,meals_planned
str,str,date,f64,f64,f64,list[struct[2]]
"""ddcd4d4a-9942-4c2a-a583-0bdf69…","""ddcd4d4a-9942-4c2a-a583-0bdf69…",2025-06-06,54.3744,828.616587,13.527439,"[{100,282.333344}, {180,325.0}, {206,190.0}]"
"""ddcd4d4a-9942-4c2a-a583-0bdf69…","""ddcd4d4a-9942-4c2a-a583-0bdf69…",2025-06-03,40.029025,886.919327,13.527439,"[{95,239.0}, {94,209.666672}, {167,272.5}]"
"""ddcd4d4a-9942-4c2a-a583-0bdf69…","""ddcd4d4a-9942-4c2a-a583-0bdf69…",2025-06-02,39.272509,833.382401,13.527439,"[{32,536.666687}, {49,288.0}, {490,190.0}]"
"""ddcd4d4a-9942-4c2a-a583-0bdf69…","""ddcd4d4a-9942-4c2a-a583-0bdf69…",2025-06-04,40.609657,903.47806,13.527439,"[{8,326.666656}, {50,239.333328}, {249,190.0}]"
"""ddcd4d4a-9942-4c2a-a583-0bdf69…","""ddcd4d4a-9942-4c2a-a583-0bdf69…",2025-06-05,46.893462,846.033886,13.527439,"[{75,239.0}, {117,211.0}, {410,400.5}]"


In [24]:
dim_meals_planned = (
    menus_week
    .select(pl.col('id').alias('menu_id'), 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')
)

dim_meals_planned.head()

menu_id,meal,pos
str,i64,f32
"""ddcd4d4a-9942-4c2a-a583-0bdf69…",100,282.333344
"""ddcd4d4a-9942-4c2a-a583-0bdf69…",180,325.0
"""ddcd4d4a-9942-4c2a-a583-0bdf69…",206,190.0
"""ddcd4d4a-9942-4c2a-a583-0bdf69…",95,239.0
"""ddcd4d4a-9942-4c2a-a583-0bdf69…",94,209.666672
